In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_path = "/content/drive/Othercomputers"
# Updated to .mov based on your screenshot
target_file = "Karwan Bazar Pedestrians.mp4" 

for root, dirs, files in os.walk(base_path):
    if target_file in files:
        exact_path = os.path.join(root, target_file)
        print("\n👇 Highlight and copy the path below 👇\n")
        print(exact_path)
        break


👇 Highlight and copy the path below 👇

/content/drive/Othercomputers/My Mac/Documents/Ml_project/Ho Chi Minh City Traffic Intersection Vietnam.mov


In [ ]:
%pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 617.5 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
import  cv2 
import numpy as np 
import pandas as pd 
from ultralytics import YOLO
import os 

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


#Feature Extraction 

In [ ]:
"we are loading model to gpu if available"
def initialize_yolo(model_size='yolov8s.pt'):
    model = YOLO(model_size)
    return model 


In [ ]:
# taking the raw video and applying masking,blurring cropping , cropping 
#running the yolo detection/tracting and saving the frame by frame co-ordiantes to the csv file 

def processing_video_frame(video_path,output_csv_path="extracted_features.csv",conf_threshold=0.4):
    cap=cv2.VideoCapture(video_path)
    # for safety precaution 
    if not cap.isOpened():
        print("error:Could not open video {video_path}")
        return 
    # getting the video properties 
    width= int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps=cap.get(cv2.CAP_PROP_FPS)
    total_frame=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Resolution : {width}x{height} | fps : {fps} | Total Frame {total_frame}")
    model = initialize_yolo('yolov8s.pt')

    # tagert classes in coco dataset : 0 :person , 2: car ,3:motorcycle ,5:bus 7:truck - why all prime 
    target_classes=[0,2,3,5,7]
    frame_data=[]
    frame_idx=0

    while cap.isOpened():
        ret , frame = cap.read()
        if not ret:
            break 
        # Preprocessing Pipeline do it later 


        # Object  Detection & Tracking (yolo)
        # using . track() maintains unique id accross the frame (ByteTrack/Bot-sort)--- learn what is that 
        results= model.track(frame,persist=True,verbose=False,conf=conf_threshold)
        if results[0].boxes is not None and results[0].boxes.id is not None :
            boxes=results[0].boxes.xyxy.cpu().numpy()
            track_ids=results[0].boxes.id.cpu().numpy()
            confidences= results[0].boxes.conf.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy()

            for box , track_id , conf,cls in zip(boxes,track_ids,confidences,classes):
                if int(cls) in target_classes:
                    x1,y1,x2,y2=map(int,box)
                    class_name=model.names[int(cls)]
                    frame_data.append({
                        'frame_idx':frame_idx,
                        'track_id':int(track_id),
                        'class':class_name,
                        'confidence':float(conf),
                        'x1':x1,
                        'y1':y1,
                        'x2':x2,
                        'y2':y2


                    })
        frame_idx+=1
        if frame_idx%50==0:
            print(f"Processed {frame_idx}/{total_frame} frames ........")
        cap.release()

    df=pd.DataFrame(frame_data)
    df.to_csv(output_csv_path,index=False)
    return df 







         





In [ ]:
video_path=r"/content/drive/Othercomputers/My Mac/Documents/Ml_project/Ho Chi Minh City Traffic Intersection Vietnam.mov"

extracted_df=processing_video_frame(video_path)
display(extracted_df.head())


Resolution : 1920x1080 | fps : 30.0 | Total Frame 465
we are loading the yolo yolov8s.pt
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 793ms
Prepared 1 package in 323ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



,frame_idx,track_id,class,confidence,x1,y1,x2,y2
0,0,1,bus,0.935800,934,216,1267,420
1,0,2,car,0.843455,0,602,173,801
2,0,3,car,0.824179,772,232,911,303
3,0,4,car,0.821313,491,299,618,375
4,0,5,car,0.808416,438,372,598,485
